# 3RScan Temporal CLIP Embedding Dataset Builder

This notebook builds training data for your residual model by:

1. Rendering instance masks from the segmented mesh (`labels.instances.annotated.v2.ply`) for each RGB frame pose.
2. Aligning rendered instance masks with original RGB images.
3. Extracting masked object crops.
4. Computing CLIP embeddings for each visible object per frame.
5. Saving per-scan JSON files grouped by `scan_id -> object_id -> global_id`.

The code is designed to run on many scans (for example 600 scans) with checkpoint-friendly per-scan outputs.

In [1]:
import io
import os
import re
import json
import math
import shutil
import tempfile
import zipfile
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import cv2
import torch
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

import clip

from pytorch3d.io import IO
from pytorch3d.structures import Meshes
from pytorch3d.renderer import (
    RasterizationSettings,
    MeshRasterizer,
    TexturesVertex,
)
from pytorch3d.renderer.cameras import PerspectiveCameras
from pytorch3d.ops import interpolate_face_attributes


@dataclass
class BuildConfig:
    dataset_root: Path
    objects_json_path: Path
    output_root: Path
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    renderer_backend: str = "pyrender"  # "pyrender" (OpenGL offscreen) or "pytorch3d"
    renderer_device: str = "cpu"
    clip_model_name: str = "ViT-B/32"
    clip_batch_size: int = 128
    min_visible_pixels: int = 64
    frame_stride: int = 1
    max_scans: Optional[int] = None
    save_overlay_debug: bool = False
    save_comparison_debug: bool = False
    comparison_every_n_frames: int = 100
    overlay_alpha: float = 0.45
    use_nearest_color_fallback: bool = False


print(f"Using CLIP device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print("Renderer backend default: pyrender")

Using CLIP device: cuda
Renderer backend default: pyrender


In [14]:
def parse_info_txt(raw_text: str) -> dict:
    info = {}
    for line in raw_text.splitlines():
        line = line.strip()
        if not line or "=" not in line:
            continue
        k, v = line.split("=", 1)
        info[k.strip()] = v.strip()
    return info


def parse_mat4_from_text(raw_text: str) -> np.ndarray:
    vals = [float(x) for x in raw_text.replace("\n", " ").split()]
    if len(vals) != 16:
        raise ValueError(f"Expected 16 values for pose, got {len(vals)}")
    return np.asarray(vals, dtype=np.float32).reshape(4, 4)


def parse_hex_color(hex_color: str) -> Tuple[int, int, int]:
    h = hex_color.strip().lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Invalid color: {hex_color}")
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))


def rgb_to_color_key(rgb: Tuple[int, int, int]) -> int:
    r, g, b = int(rgb[0]), int(rgb[1]), int(rgb[2])
    return (r << 16) | (g << 8) | b


def color_key_to_rgb(color_key: int) -> Tuple[int, int, int]:
    return ((color_key >> 16) & 255, (color_key >> 8) & 255, color_key & 255)


def rgb_image_to_color_key_image(rgb_img: np.ndarray) -> np.ndarray:
    # Encode RGB triplets to one int32 channel for faster unique/mask operations.
    r = rgb_img[:, :, 0].astype(np.int32)
    g = rgb_img[:, :, 1].astype(np.int32)
    b = rgb_img[:, :, 2].astype(np.int32)
    return (r << 16) | (g << 8) | b


def load_3dssg_objects(objects_json_path: Path) -> Dict[str, dict]:
    with open(objects_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    out = {}
    scans = data.get("scans", [])
    for scan_entry in scans:
        scan_id = scan_entry["scan"]
        objects = scan_entry.get("objects", [])
        by_obj_id = {}
        by_color = {}
        by_color_key = {}
        for obj in objects:
            object_id = str(obj.get("id"))
            global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else None
            label = str(obj.get("label")) if obj.get("label") is not None else "unknown"
            rgb = parse_hex_color(str(obj.get("ply_color", "#000000")))
            ckey = rgb_to_color_key(rgb)
            by_obj_id[object_id] = {
                "object_id": object_id,
                "global_id": global_id,
                "label": label,
                "ply_color": rgb,
            }
            by_color[rgb] = object_id
            by_color_key[ckey] = object_id

        out[scan_id] = {
            "by_obj_id": by_obj_id,
            "by_color": by_color,
            "by_color_key": by_color_key,
        }
    return out


def build_frame_index_from_dir(seq_dir: Path) -> List[dict]:
    color_pat = re.compile(r"frame-(\d{6})\.color\.jpg$")
    depth_pat = re.compile(r"frame-(\d{6})\.depth\.pgm$")
    pose_pat = re.compile(r"frame-(\d{6})\.pose\.txt$")

    color = {}
    depth = {}
    pose = {}

    for p in seq_dir.iterdir():
        if not p.is_file():
            continue
        n = p.name
        m = color_pat.search(n)
        if m:
            color[int(m.group(1))] = p
            continue
        m = depth_pat.search(n)
        if m:
            depth[int(m.group(1))] = p
            continue
        m = pose_pat.search(n)
        if m:
            pose[int(m.group(1))] = p

    frame_ids = sorted(set(color.keys()) & set(pose.keys()))
    return [
        {
            "frame_idx": fi,
            "color_path": color[fi],
            "depth_path": depth.get(fi),
            "pose_path": pose[fi],
        }
        for fi in frame_ids
    ]


def read_image_from_path(path: Path) -> np.ndarray:
    img_bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise RuntimeError(f"Failed to decode image {path}")
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def read_pose_from_path(path: Path) -> np.ndarray:
    with open(path, "r", encoding="utf-8") as f:
        raw = f.read()
    return parse_mat4_from_text(raw)


def overlay_mask(rgb: np.ndarray, mask: np.ndarray, color: Tuple[int, int, int], alpha: float = 0.45) -> np.ndarray:
    out = rgb.copy()
    c = np.asarray(color, dtype=np.float32).reshape(1, 1, 3)
    blend = (1.0 - alpha) * out.astype(np.float32) + alpha * c
    out[mask] = blend[mask].astype(np.uint8)
    return out


def make_rgb_render_comparison(rgb: np.ndarray, rendered_rgb: np.ndarray, overlay_rgb: np.ndarray) -> np.ndarray:
    h, w = rgb.shape[:2]
    if rendered_rgb.shape[:2] != (h, w):
        rendered_rgb = cv2.resize(rendered_rgb, (w, h), interpolation=cv2.INTER_NEAREST)
    if overlay_rgb.shape[:2] != (h, w):
        overlay_rgb = cv2.resize(overlay_rgb, (w, h), interpolation=cv2.INTER_NEAREST)

    panel = np.concatenate([rgb, rendered_rgb, overlay_rgb], axis=1)
    strip = np.zeros((32, panel.shape[1], 3), dtype=np.uint8)
    cv2.putText(strip, "RGB", (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(strip, "Rendered Instance", (w + 10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    cv2.putText(strip, "Overlay", (2 * w + 10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 255), 2, cv2.LINE_AA)
    return np.concatenate([strip, panel], axis=0)


def mask_to_crop(rgb: np.ndarray, mask: np.ndarray) -> Tuple[np.ndarray, List[int], int]:
    ys, xs = np.where(mask)
    if ys.size == 0:
        return None, None, 0
    y0, y1 = int(ys.min()), int(ys.max())
    x0, x1 = int(xs.min()), int(xs.max())

    masked = np.zeros_like(rgb)
    masked[mask] = rgb[mask]
    crop = masked[y0:y1 + 1, x0:x1 + 1]
    return crop, [x0, y0, x1, y1], int(mask.sum())

In [15]:
class SegmentedMeshRenderer:
    """PyTorch3D fallback renderer."""

    def __init__(self, mesh_path: Path, device: str = "cpu"):
        io_loader = IO()
        mesh = io_loader.load_mesh(str(mesh_path))
        if mesh.isempty():
            raise RuntimeError(f"Empty mesh: {mesh_path}")

        if mesh.textures is None or not hasattr(mesh.textures, "verts_features_list"):
            raise RuntimeError(
                f"Mesh at {mesh_path} does not provide vertex colors needed for object IDs"
            )

        self.device = torch.device(device)
        verts = [v.to(self.device) for v in mesh.verts_list()]
        faces = [f.to(self.device) for f in mesh.faces_list()]
        feats = [c.to(self.device) for c in mesh.textures.verts_features_list()]
        textures = TexturesVertex(verts_features=feats)
        self.mesh = Meshes(verts=verts, faces=faces, textures=textures)

    @staticmethod
    def _twc_to_world_to_camera(t_wc: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        r_wc = t_wc[:3, :3]
        t_wc_vec = t_wc[:3, 3]
        r_cw = r_wc.T
        t_cw = -r_cw @ t_wc_vec
        return r_cw, t_cw

    def render_vertex_color_image(
        self,
        t_wc: np.ndarray,
        k: np.ndarray,
        h: int,
        w: int,
    ) -> np.ndarray:
        r_cw, t_cw = self._twc_to_world_to_camera(t_wc)

        fx, fy = float(k[0, 0]), float(k[1, 1])
        cx, cy = float(k[0, 2]), float(k[1, 2])

        cameras = PerspectiveCameras(
            focal_length=torch.tensor([[fx, fy]], dtype=torch.float32, device=self.device),
            principal_point=torch.tensor([[cx, cy]], dtype=torch.float32, device=self.device),
            image_size=torch.tensor([[h, w]], dtype=torch.float32, device=self.device),
            in_ndc=False,
            R=torch.tensor(r_cw, dtype=torch.float32, device=self.device).unsqueeze(0),
            T=torch.tensor(t_cw, dtype=torch.float32, device=self.device).unsqueeze(0),
        )

        raster_settings = RasterizationSettings(
            image_size=(h, w),
            blur_radius=0.0,
            faces_per_pixel=1,
            perspective_correct=True,
            cull_backfaces=False,
        )
        rasterizer = MeshRasterizer(cameras=cameras, raster_settings=raster_settings)
        fragments = rasterizer(self.mesh)

        pix_to_face = fragments.pix_to_face
        bary = fragments.bary_coords

        verts_rgb = self.mesh.textures.verts_features_packed()
        faces = self.mesh.faces_packed()
        face_rgb = verts_rgb[faces].unsqueeze(0)

        sampled = interpolate_face_attributes(pix_to_face, bary, face_rgb)
        rgb = sampled[0, ..., 0, :]

        valid = (pix_to_face[0, ..., 0] >= 0).unsqueeze(-1)
        rgb = torch.where(valid, rgb, torch.zeros_like(rgb))
        rgb_u8 = torch.clamp(rgb * 255.0, 0, 255).byte().cpu().numpy()
        return rgb_u8


class OpenGLSegmentedMeshRenderer:
    """Fast OpenGL offscreen renderer using pyrender."""

    def __init__(self, mesh_path: Path, width: int, height: int):
        try:
            import trimesh
            import pyrender
        except Exception as e:
            raise RuntimeError(
                "OpenGL renderer requires trimesh and pyrender. Install with: pip install trimesh pyrender"
            ) from e

        self.trimesh = trimesh
        self.pyrender = pyrender

        tm = trimesh.load(str(mesh_path), process=False)
        if isinstance(tm, trimesh.Scene):
            if len(tm.geometry) == 0:
                raise RuntimeError(f"Empty mesh scene: {mesh_path}")
            tm = trimesh.util.concatenate(tuple(tm.geometry.values()))

        if not hasattr(tm.visual, "vertex_colors") or tm.visual.vertex_colors is None:
            raise RuntimeError(f"Mesh at {mesh_path} does not provide vertex colors needed for object IDs")

        self.scene = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 0.0], ambient_light=[1.0, 1.0, 1.0, 1.0])
        self.mesh = pyrender.Mesh.from_trimesh(tm, smooth=False)
        self.mesh_node = self.scene.add(self.mesh)
        self.camera_node = None

        self.width = int(width)
        self.height = int(height)
        self.renderer = pyrender.OffscreenRenderer(viewport_width=self.width, viewport_height=self.height)

        # OpenCV camera (x right, y down, z forward) to OpenGL camera (x right, y up, z backward).
        self.cv_to_gl = np.array(
            [[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, -1, 0], [0, 0, 0, 1]],
            dtype=np.float32,
        )

    def _resize_if_needed(self, width: int, height: int) -> None:
        width = int(width)
        height = int(height)
        if width == self.width and height == self.height:
            return
        self.width = width
        self.height = height
        self.renderer.delete()
        self.renderer = self.pyrender.OffscreenRenderer(viewport_width=self.width, viewport_height=self.height)

    def render_vertex_color_image(self, t_wc: np.ndarray, k: np.ndarray, h: int, w: int) -> np.ndarray:
        self._resize_if_needed(w, h)

        fx, fy = float(k[0, 0]), float(k[1, 1])
        cx, cy = float(k[0, 2]), float(k[1, 2])

        cam = self.pyrender.IntrinsicsCamera(fx=fx, fy=fy, cx=cx, cy=cy, znear=0.01, zfar=1000.0)
        cam_pose_gl = (t_wc @ self.cv_to_gl).astype(np.float32)

        if self.camera_node is not None:
            self.scene.remove_node(self.camera_node)
        self.camera_node = self.scene.add(cam, pose=cam_pose_gl)

        flags = self.pyrender.constants.RenderFlags.FLAT
        color, _ = self.renderer.render(self.scene, flags=flags)

        self.scene.remove_node(self.camera_node)
        self.camera_node = None

        if color.ndim == 3 and color.shape[2] >= 3:
            return color[:, :, :3].astype(np.uint8)
        raise RuntimeError("OpenGL renderer returned invalid color buffer")

    def close(self) -> None:
        if self.renderer is not None:
            self.renderer.delete()
            self.renderer = None


class ClipImageEncoder:
    def __init__(self, model_name: str, device: str = "cuda"):
        self.device = device
        self.model, self.preprocess = clip.load(model_name, device=device)
        self.model.eval()

    @torch.no_grad()
    def encode_batch(self, image_rgb_list: List[np.ndarray], batch_size: int = 64) -> List[np.ndarray]:
        if len(image_rgb_list) == 0:
            return []

        out = []
        for i in range(0, len(image_rgb_list), batch_size):
            chunk = image_rgb_list[i:i + batch_size]
            batch = torch.stack([self.preprocess(Image.fromarray(img)) for img in chunk], dim=0).to(self.device)
            emb = self.model.encode_image(batch)
            emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-8)
            out.extend([e.detach().cpu().numpy().astype(np.float32) for e in emb])
        return out

    @torch.no_grad()
    def encode(self, image_rgb: np.ndarray) -> np.ndarray:
        return self.encode_batch([image_rgb])[0]

In [16]:
def nearest_object_id_from_color(rgb: Tuple[int, int, int], by_color: Dict[Tuple[int, int, int], str], max_dist: float = 6.0):
    if rgb in by_color:
        return by_color[rgb]

    c = np.asarray(rgb, dtype=np.float32)
    best_id = None
    best_d = 1e9
    for ref_rgb, oid in by_color.items():
        d = float(np.linalg.norm(c - np.asarray(ref_rgb, dtype=np.float32)))
        if d < best_d:
            best_d = d
            best_id = oid
    if best_d <= max_dist:
        return best_id
    return None


def build_color_to_object_lookup(scan_objects: dict):
    return dict(scan_objects.get("by_color", {})), dict(scan_objects.get("by_color_key", {}))


def ensure_scan_output_template(scan_id: str, scan_objects: dict) -> dict:
    objects_block = {}
    for oid, meta in scan_objects["by_obj_id"].items():
        objects_block[oid] = {
            "object_id": oid,
            "global_id": meta["global_id"],
            "label": meta["label"],
            "embeddings": [],
        }
    return {
        "scan_id": scan_id,
        "objects": objects_block,
    }


def process_single_scan(
    scan_dir: Path,
    scan_objects_map: Dict[str, dict],
    cfg: BuildConfig,
    encoder: ClipImageEncoder,
) -> Optional[Path]:
    scan_id = scan_dir.name
    if scan_id not in scan_objects_map:
        print(f"[SKIP] {scan_id}: not found in 3DSSG objects metadata")
        return None

    sequence_zip = scan_dir / "sequence.zip"
    sequence_dir = scan_dir / "sequence"
    labels_ply = scan_dir / "labels.instances.annotated.v2.ply"

    if not labels_ply.exists():
        print(f"[SKIP] {scan_id}: missing labels.instances.annotated.v2.ply")
        return None

    # Check if already processed in a previous run
    out_scans_dir = cfg.output_root / "scans"
    out_path = out_scans_dir / f"{scan_id}.json"
    if out_path.exists():
        print(f"[SKIP] {scan_id}: already processed")
        return out_path

    has_sequence_dir = sequence_dir.exists() and sequence_dir.is_dir()
    has_sequence_zip = sequence_zip.exists()

    if not has_sequence_dir and not has_sequence_zip:
        print(f"[SKIP] {scan_id}: missing both sequence/ and sequence.zip")
        return None

    scan_objects = scan_objects_map[scan_id]
    by_color, by_color_key = build_color_to_object_lookup(scan_objects)
    result = ensure_scan_output_template(scan_id, scan_objects)

    debug_dir = cfg.output_root / "debug_overlays" / scan_id
    comp_dir = cfg.output_root / "debug_comparison" / scan_id
    tmp_root = cfg.output_root / "_tmp_sequences"
    tmp_root.mkdir(parents=True, exist_ok=True)

    if cfg.save_overlay_debug:
        debug_dir.mkdir(parents=True, exist_ok=True)
    if cfg.save_comparison_debug:
        comp_dir.mkdir(parents=True, exist_ok=True)

    def run_on_sequence_dir(seq_root: Path) -> None:
        info_path = seq_root / "_info.txt"
        if not info_path.exists():
            print(f"[SKIP] {scan_id}: sequence folder has no _info.txt")
            return

        with open(info_path, "r", encoding="utf-8") as f:
            info = parse_info_txt(f.read())

        color_w = int(info["m_colorWidth"])
        color_h = int(info["m_colorHeight"])
        k_color_vals = [float(x) for x in info["m_calibrationColorIntrinsic"].split()]
        k_color = np.asarray(k_color_vals, dtype=np.float32).reshape(4, 4)[:3, :3]

        backend = str(cfg.renderer_backend).lower()
        if backend in ("pyrender", "opengl"):
            renderer = OpenGLSegmentedMeshRenderer(labels_ply, width=color_w, height=color_h)
        else:
            renderer = SegmentedMeshRenderer(labels_ply, device=cfg.renderer_device)

        frames = build_frame_index_from_dir(seq_root)
        if cfg.frame_stride > 1:
            frames = frames[:: cfg.frame_stride]

        render_fail_count = 0

        try:
            for fr in tqdm(frames, desc=f"scan {scan_id}", leave=False):
                frame_idx = int(fr["frame_idx"])
                rgb = read_image_from_path(fr["color_path"])
                t_wc = read_pose_from_path(fr["pose_path"])

                if not np.isfinite(t_wc).all():
                    continue

                try:
                    rendered_obj_rgb = renderer.render_vertex_color_image(
                        t_wc=t_wc,
                        k=k_color,
                        h=color_h,
                        w=color_w,
                    )
                except Exception as e:
                    render_fail_count += 1
                    if render_fail_count <= 5:
                        print(f"[WARN] {scan_id} frame {frame_idx}: render failed: {e}")
                    continue

                if rendered_obj_rgb.shape[:2] != rgb.shape[:2]:
                    rendered_obj_rgb = cv2.resize(
                        rendered_obj_rgb,
                        (rgb.shape[1], rgb.shape[0]),
                        interpolation=cv2.INTER_NEAREST,
                    )

                # Fast path: encode RGB colors to one int32 channel.
                color_key_img = rgb_image_to_color_key_image(rendered_obj_rgb)
                uniq_keys, uniq_counts = np.unique(color_key_img, return_counts=True)

                frame_objects = []
                overlay_img = rgb.copy()

                for ckey, count in zip(uniq_keys.tolist(), uniq_counts.tolist()):
                    ckey = int(ckey)
                    if ckey == 0:
                        continue
                    if count < int(cfg.min_visible_pixels):
                        continue

                    oid = by_color_key.get(ckey)
                    if oid is None and cfg.use_nearest_color_fallback:
                        rgb_tuple = color_key_to_rgb(ckey)
                        oid = nearest_object_id_from_color(rgb_tuple, by_color)
                    if oid is None:
                        continue

                    mask = color_key_img == ckey
                    crop, bbox, px = mask_to_crop(rgb, mask)
                    if crop is None or crop.size == 0:
                        continue

                    frame_objects.append(
                        {
                            "object_id": oid,
                            "color_key": ckey,
                            "mask": mask,
                            "crop": crop,
                            "bbox": bbox,
                            "visible_pixels": px,
                            "frame_name": fr["color_path"].name,
                            "frame_idx": frame_idx,
                        }
                    )

                    if cfg.save_overlay_debug or cfg.save_comparison_debug:
                        overlay_img = overlay_mask(
                            overlay_img,
                            mask,
                            color_key_to_rgb(ckey),
                            alpha=cfg.overlay_alpha,
                        )

                if len(frame_objects) > 0:
                    batch_crops = [item["crop"] for item in frame_objects]
                    batch_embs = encoder.encode_batch(batch_crops, batch_size=int(cfg.clip_batch_size))

                    for item, emb in zip(frame_objects, batch_embs):
                        obj_block = result["objects"][item["object_id"]]
                        obj_block["embeddings"].append(
                            {
                                "frame_index": item["frame_idx"],
                                "frame_name": item["frame_name"],
                                "bbox_xyxy": item["bbox"],
                                "visible_pixels": item["visible_pixels"],
                                "embedding": emb.tolist(),
                            }
                        )

                if cfg.save_overlay_debug:
                    out_dbg = debug_dir / f"frame-{frame_idx:06d}.png"
                    Image.fromarray(overlay_img).save(out_dbg)

                if cfg.save_comparison_debug and (frame_idx % max(1, int(cfg.comparison_every_n_frames)) == 0):
                    comparison = make_rgb_render_comparison(rgb, rendered_obj_rgb, overlay_img)
                    comp_path = comp_dir / f"frame-{frame_idx:06d}.png"
                    Image.fromarray(comparison).save(comp_path)

            if render_fail_count > 0:
                print(f"[WARN] {scan_id}: total render failures: {render_fail_count}/{len(frames)}")
        finally:
            if hasattr(renderer, "close"):
                try:
                    renderer.close()
                except Exception:
                    pass

    if has_sequence_dir:
        run_on_sequence_dir(sequence_dir)
    else:
        try:
            with tempfile.TemporaryDirectory(prefix=f"seq_{scan_id}_", dir=str(tmp_root)) as tmp_dir_str:
                tmp_dir = Path(tmp_dir_str)
                try:
                    with zipfile.ZipFile(sequence_zip, "r") as zf:
                        zf.extractall(tmp_dir)
                except (zipfile.BadZipFile, OSError, RuntimeError) as e:
                    print(f"[SKIP] {scan_id}: corrupted or invalid sequence.zip - {type(e).__name__}: {e}")
                    return None
                run_on_sequence_dir(tmp_dir)
        except Exception as e:
            print(f"[SKIP] {scan_id}: failed to process sequence.zip - {type(e).__name__}: {e}")
            return None

    out_scans_dir.mkdir(parents=True, exist_ok=True)

    kept = {
        oid: odata
        for oid, odata in result["objects"].items()
        if len(odata["embeddings"]) > 0
    }
    result["objects"] = kept

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False)

    return out_path


def build_dataset(cfg: BuildConfig):
    cfg.output_root.mkdir(parents=True, exist_ok=True)

    scan_objects_map = load_3dssg_objects(cfg.objects_json_path)
    all_scan_dirs = [p for p in cfg.dataset_root.iterdir() if p.is_dir()]
    all_scan_dirs = sorted(all_scan_dirs, key=lambda p: p.name)

    if cfg.max_scans is not None:
        all_scan_dirs = all_scan_dirs[: cfg.max_scans]

    valid_scan_dirs = []
    skipped_missing_ply = 0
    skipped_missing_sequence = 0
    for scan_dir in all_scan_dirs:
        has_seq_dir = (scan_dir / "sequence").exists()
        has_seq_zip = (scan_dir / "sequence.zip").exists()
        if not has_seq_dir and not has_seq_zip:
            skipped_missing_sequence += 1
            continue
        if not (scan_dir / "labels.instances.annotated.v2.ply").exists():
            skipped_missing_ply += 1
            continue
        valid_scan_dirs.append(scan_dir)

    print(f"Scans discovered: {len(all_scan_dirs)}")
    print(f"Scans skipped (missing sequence folder/zip): {skipped_missing_sequence}")
    print(f"Scans skipped (missing labels.instances.annotated.v2.ply): {skipped_missing_ply}")
    print(f"Scans to process: {len(valid_scan_dirs)}")

    encoder = ClipImageEncoder(cfg.clip_model_name, device=cfg.device)

    outputs = []
    for scan_dir in tqdm(valid_scan_dirs, desc="all scans"):
        out_path = process_single_scan(
            scan_dir=scan_dir,
            scan_objects_map=scan_objects_map,
            cfg=cfg,
            encoder=encoder,
        )
        if out_path is not None:
            outputs.append(out_path)

    manifest = {
        "num_scans_discovered": len(all_scan_dirs),
        "num_scans_filtered": len(valid_scan_dirs),
        "num_skipped_missing_sequence": skipped_missing_sequence,
        "num_skipped_missing_labels_ply": skipped_missing_ply,
        "num_scans_written": len(outputs),
        "scan_files": [str(p) for p in outputs],
    }
    manifest_path = cfg.output_root / "manifest.json"
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

    tmp_root = cfg.output_root / "_tmp_sequences"
    if tmp_root.exists():
        shutil.rmtree(tmp_root, ignore_errors=True)

    print(f"Done. Wrote {len(outputs)} scan files.")
    print(f"Manifest: {manifest_path}")
    return manifest_path

In [17]:
# Configure paths for your workspace.
cfg = BuildConfig(
    dataset_root=Path("/home/station2/Zermat/RTSGS-SLAM/Datasets/3Rscan/scans"),
    objects_json_path=Path("/home/station2/Zermat/RTSGS-SLAM/Datasets/3DSSG/objects.json"),
    output_root=Path("/home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset"),
    frame_stride=1,
    min_visible_pixels=128,
    max_scans=None,  # set to a small number like 2 for smoke testing
    save_overlay_debug=False,
)

print(cfg)

BuildConfig(dataset_root=PosixPath('/home/station2/Zermat/RTSGS-SLAM/Datasets/3Rscan/scans'), objects_json_path=PosixPath('/home/station2/Zermat/RTSGS-SLAM/Datasets/3DSSG/objects.json'), output_root=PosixPath('/home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset'), device='cuda', renderer_backend='pyrender', renderer_device='cpu', clip_model_name='ViT-B/32', clip_batch_size=128, min_visible_pixels=128, frame_stride=1, max_scans=None, save_overlay_debug=False, save_comparison_debug=False, comparison_every_n_frames=100, overlay_alpha=0.45, use_nearest_color_fallback=False)


In [18]:
# Run dataset build.
# Important: this requires labels.instances.annotated.v2.ply for each scan.
manifest_path = build_dataset(cfg)
manifest_path

Scans discovered: 1482
Scans skipped (missing sequence folder/zip): 0
Scans skipped (missing labels.instances.annotated.v2.ply): 101
Scans to process: 1381


all scans:   0%|          | 0/1381 [00:00<?, ?it/s]

[SKIP] 00d42bed-778d-2ac6-86a7-0e0e5f5f5660: already processed
[SKIP] 02b33df9-be2b-2d54-9062-1253be3ce186: already processed
[SKIP] 02b33dfb-be2b-2d54-92d2-cd012b2b3c40: already processed
[SKIP] 02b33dfd-be2b-2d54-91d2-55454852009e: already processed
[SKIP] 02b33e01-be2b-2d54-93fb-4145a709cec5: already processed
[SKIP] 02b33e03-be2b-2d54-9129-5d28efdd68fa: already processed
[SKIP] 05c6ede7-2e69-23b1-8b27-c1cb868f1938: already processed
[SKIP] 095821f7-e2c2-2de1-9568-b9ce59920e29: already processed
[SKIP] 095821f9-e2c2-2de1-9707-8f735cd1c148: already processed
[SKIP] 095821fb-e2c2-2de1-94df-20f2cb423bcb: already processed
[SKIP] 09582205-e2c2-2de1-9475-1cdac7639e60: already processed
[SKIP] 09582207-e2c2-2de1-972c-225d968c2ab4: already processed
[SKIP] 09582209-e2c2-2de1-9610-08baed932919: already processed
[SKIP] 0958220b-e2c2-2de1-96bc-739f09c1e8f8: already processed
[SKIP] 0958220d-e2c2-2de1-9710-c37018da1883: already processed
[SKIP] 09582212-e2c2-2de1-9700-fa44b14fbded: already pr

scan cf00577c-b8f0-2aa8-869e-175d8b655d12:   0%|          | 0/630 [00:00<?, ?it/s]

scan cf00577e-b8f0-2aa8-8751-be34df30341f:   0%|          | 0/620 [00:00<?, ?it/s]

...

scan feefd90b-9d00-2ce5-8119-b936443cc39b:   0%|          | 0/334 [00:00<?, ?it/s]

scan ffa41872-6f78-2040-8600-7b90c9a0fa89:   0%|          | 0/153 [00:00<?, ?it/s]

scan ffa41874-6f78-2040-85a8-8056ac60c764:   0%|          | 0/206 [00:00<?, ?it/s]

Done. Wrote 1376 scan files.
Manifest: /home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset/manifest.json


PosixPath('/home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset/manifest.json')

### Training

In [2]:
from typing import Iterable

# Paths for residual-model training data
clip_scans_root = Path("/home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset/scans")
target_json_path = Path("/home/station2/Zermat/RTSGS-SLAM/SceneGraph/clean_data_all_scans.json")

assert clip_scans_root.exists(), f"Missing clip scans dir: {clip_scans_root}"
assert target_json_path.exists(), f"Missing target json: {target_json_path}"

print("clip_scans_root:", clip_scans_root)
print("target_json_path:", target_json_path)
print("num per-scan files:", len(list(clip_scans_root.glob("*.json"))))


def iter_clip_scan_files(scans_root: Path) -> Iterable[Path]:
    for p in sorted(scans_root.glob("*.json")):
        if p.is_file():
            yield p


def load_clip_scan_json(scan_json_path: Path) -> dict:
    with open(scan_json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def normalize_vec(v: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    n = float(np.linalg.norm(v))
    if n < eps:
        return v.astype(np.float32)
    return (v / n).astype(np.float32)


def collect_required_object_keys(
    scans_root: Path,
    max_scans: Optional[int] = None,
    min_seq_len: int = 2,
) -> set:
    required = set()
    files = list(iter_clip_scan_files(scans_root))
    if max_scans is not None:
        files = files[: int(max_scans)]

    for scan_file in tqdm(files, desc="collect required keys"):
        d = load_clip_scan_json(scan_file)
        scan_id = str(d.get("scan_id", scan_file.stem))
        for object_id, obj in d.get("objects", {}).items():
            embs = obj.get("embeddings", [])
            if len(embs) < int(min_seq_len):
                continue
            global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else "None"
            required.add((scan_id, str(object_id), global_id))
    return required

clip_scans_root: /home/station2/Zermat/RTSGS-SLAM/SceneGraph/clip_embedding_dataset/scans
target_json_path: /home/station2/Zermat/RTSGS-SLAM/SceneGraph/clean_data_all_scans.json
num per-scan files: 1376


In [3]:
def build_target_embedding_map(
    target_path: Path,
    required_keys: set,
    use_ijson: bool = True,
) -> Dict[Tuple[str, str, str], np.ndarray]:
    """Build only needed (scan_id, object_id, global_id) -> canonical embedding map.

    This avoids loading the full 1.5GB target JSON into memory.
    """
    target_map: Dict[Tuple[str, str, str], np.ndarray] = {}

    if use_ijson:
        try:
            ijson = __import__("ijson")
            with open(target_path, "rb") as f:
                for scan in ijson.items(f, "scans.item"):
                    scan_id = str(scan.get("scan"))
                    objects = scan.get("objects", [])
                    for obj in objects:
                        object_id = str(obj.get("id"))
                        global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else "None"
                        key = (scan_id, object_id, global_id)
                        if key not in required_keys:
                            continue

                        emb = obj.get("clip_embedding")
                        if emb is None:
                            continue
                        arr = np.asarray(emb, dtype=np.float32)
                        target_map[key] = normalize_vec(arr)

                        if len(target_map) == len(required_keys):
                            return target_map
            return target_map
        except ImportError:
            print("[WARN] ijson not installed; falling back to full JSON load.")
            print("       Install ijson for streaming on large files: pip install ijson")

    with open(target_path, "r", encoding="utf-8") as f:
        full = json.load(f)
    for scan in full.get("scans", []):
        scan_id = str(scan.get("scan"))
        for obj in scan.get("objects", []):
            object_id = str(obj.get("id"))
            global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else "None"
            key = (scan_id, object_id, global_id)
            if key not in required_keys:
                continue
            emb = obj.get("clip_embedding")
            if emb is None:
                continue
            target_map[key] = normalize_vec(np.asarray(emb, dtype=np.float32))
    return target_map


def build_temporal_training_records(
    scans_root: Path,
    target_map: Dict[Tuple[str, str, str], np.ndarray],
    max_scans: Optional[int] = None,
    min_seq_len: int = 2,
) -> List[dict]:
    records: List[dict] = []
    files = list(iter_clip_scan_files(scans_root))
    if max_scans is not None:
        files = files[: int(max_scans)]

    for scan_file in tqdm(files, desc="build training records"):
        d = load_clip_scan_json(scan_file)
        scan_id = str(d.get("scan_id", scan_file.stem))
        objects = d.get("objects", {})

        for object_id, obj in objects.items():
            global_id = str(obj.get("global_id")) if obj.get("global_id") is not None else "None"
            key = (scan_id, str(object_id), global_id)
            if key not in target_map:
                continue

            seq_entries = obj.get("embeddings", [])
            if len(seq_entries) < int(min_seq_len):
                continue

            seq_entries = sorted(seq_entries, key=lambda x: int(x.get("frame_index", 0)))
            seq = []
            frame_idx = []
            for e in seq_entries:
                emb = e.get("embedding")
                if emb is None:
                    continue
                seq.append(normalize_vec(np.asarray(emb, dtype=np.float32)))
                frame_idx.append(int(e.get("frame_index", -1)))

            if len(seq) < int(min_seq_len):
                continue

            records.append(
                {
                    "scan_id": scan_id,
                    "object_id": str(object_id),
                    "global_id": global_id,
                    "frames": frame_idx,
                    "sequence": np.stack(seq, axis=0),
                    "target": target_map[key],
                }
            )
    return records


# For first pass, limit scans if needed. Set to None for full dataset.
max_scans_for_index = None
required_keys = collect_required_object_keys(
    clip_scans_root,
    max_scans=max_scans_for_index,
    min_seq_len=2,
)
print("required keys:", len(required_keys))

target_map = build_target_embedding_map(
    target_json_path,
    required_keys=required_keys,
    use_ijson=True,
)
print("matched target keys:", len(target_map))

records = build_temporal_training_records(
    clip_scans_root,
    target_map=target_map,
    max_scans=max_scans_for_index,
    min_seq_len=2,
)
print("num training records:", len(records))

if len(records) > 0:
    print("example record keys:", records[0].keys())
    print("sequence shape:", records[0]["sequence"].shape)
    print("target shape:", records[0]["target"].shape)

collect required keys:   0%|          | 0/1376 [00:00<?, ?it/s]

required keys: 39634
matched target keys: 38090


build training records:   0%|          | 0/1376 [00:00<?, ?it/s]

num training records: 38090
example record keys: dict_keys(['scan_id', 'object_id', 'global_id', 'frames', 'sequence', 'target'])
sequence shape: (50, 512)
target shape: (512,)


In [4]:
class TemporalResidualDataset(torch.utils.data.Dataset):
    def __init__(self, records: List[dict]):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int):
        r = self.records[idx]
        seq = torch.from_numpy(r["sequence"]).float()  # [T, D]
        tgt = torch.from_numpy(r["target"]).float()    # [D]
        return {
            "sequence": seq,
            "target": tgt,
            "scan_id": r["scan_id"],
            "object_id": r["object_id"],
            "global_id": r["global_id"],
        }


def pad_collate(batch: List[dict]):
    lens = [int(x["sequence"].shape[0]) for x in batch]
    d = int(batch[0]["sequence"].shape[1])
    t_max = max(lens)

    seq_padded = torch.zeros(len(batch), t_max, d, dtype=torch.float32)
    target = torch.zeros(len(batch), d, dtype=torch.float32)
    mask = torch.zeros(len(batch), t_max, dtype=torch.bool)

    meta = []
    for i, item in enumerate(batch):
        t = item["sequence"].shape[0]
        seq_padded[i, :t] = item["sequence"]
        target[i] = item["target"]
        mask[i, :t] = True
        meta.append((item["scan_id"], item["object_id"], item["global_id"]))

    return {
        "sequence": seq_padded,  # [B, T, D]
        "target": target,        # [B, D]
        "mask": mask,            # [B, T]
        "meta": meta,
    }


if len(records) == 0:
    raise RuntimeError("No records built. Increase max_scans_for_index or inspect source JSON files.")

dataset = TemporalResidualDataset(records)
loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=pad_collate,
)

batch = next(iter(loader))
print("sequence:", tuple(batch["sequence"].shape))
print("target:", tuple(batch["target"].shape))
print("mask:", tuple(batch["mask"].shape))

sequence: (16, 106, 512)
target: (16, 512)
mask: (16, 106)


In [ ]:
class GRUResidualUpdater(torch.nn.Module):
    def __init__(self, emb_dim: int, hidden_dim: int = 512, dropout: float = 0.1):
        super().__init__()
        self.input_proj = torch.nn.Linear(emb_dim * 2, hidden_dim)
        self.gru_cell = torch.nn.GRUCell(hidden_dim, hidden_dim)
        self.output_proj = torch.nn.Linear(hidden_dim, emb_dim)
        self.dropout = torch.nn.Dropout(dropout)

    def forward(self, prev_e: torch.Tensor, obs_e: torch.Tensor, hidden: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = torch.cat([prev_e, obs_e], dim=-1)
        x = torch.nn.functional.gelu(self.input_proj(x))
        hidden = self.gru_cell(x, hidden)
        hidden = self.dropout(hidden)
        residual = self.output_proj(hidden)
        return residual, hidden


def cosine_distance(a: torch.Tensor, b: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    a = a / a.norm(dim=-1, keepdim=True).clamp_min(eps)
    b = b / b.norm(dim=-1, keepdim=True).clamp_min(eps)
    return 1.0 - (a * b).sum(dim=-1)


def contrastive_alignment_loss(
    pred: torch.Tensor,
    target: torch.Tensor,
    temperature: float = 0.07,
) -> torch.Tensor:
    pred = pred / pred.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    target = target / target.norm(dim=-1, keepdim=True).clamp_min(1e-8)
    logits = pred @ target.t() / temperature
    labels = torch.arange(pred.shape[0], device=pred.device)
    loss_i2t = torch.nn.functional.cross_entropy(logits, labels)
    loss_t2i = torch.nn.functional.cross_entropy(logits.t(), labels)
    return 0.5 * (loss_i2t + loss_t2i)


def residual_temporal_loss(
    model: torch.nn.Module,
    seq: torch.Tensor,      # [B, T, D] noisy observations \tilde{e}_t
    target: torch.Tensor,   # [B, D] canonical target e*
    mask: torch.Tensor,     # [B, T] valid time steps
    lambda_step: float = 0.4,
    lambda_res: float = 1e-4,
    lambda_contrastive: float = 0.2,
    init_from_first: bool = True,
) -> Tuple[torch.Tensor, Dict[str, float], torch.Tensor]:
    b, t, d = seq.shape
    device = seq.device

    if init_from_first:
        e_prev = seq[:, 0, :].clone()
        start_t = 1
    else:
        e_prev = torch.zeros(b, d, device=device, dtype=seq.dtype)
        start_t = 0

    hidden = torch.zeros(b, model.gru_cell.hidden_size, device=device, dtype=seq.dtype)

    step_losses = []
    residual_norms = []

    for ti in range(start_t, t):
        valid = mask[:, ti]
        obs = seq[:, ti, :]
        r_t, hidden = model(e_prev, obs, hidden)
        e_new = e_prev + r_t

        if valid.any():
            step_loss = cosine_distance(e_new[valid], target[valid]).mean()
            step_losses.append(step_loss)
            scale = torch.maximum(
                e_prev[valid].norm(dim=-1),
                obs[valid].norm(dim=-1),
            ).detach().clamp_min(1.0)
            residual_norms.append((r_t[valid].norm(dim=-1) / scale).pow(2))

        e_prev = torch.where(valid.unsqueeze(-1), e_new, e_prev)

    final_loss = cosine_distance(e_prev, target).mean()
    contrastive_loss = contrastive_alignment_loss(e_prev, target)

    if len(step_losses) > 0:
        l_step = torch.stack(step_losses).mean()
        l_res = torch.cat(residual_norms).mean()
    else:
        l_step = torch.tensor(0.0, device=device)
        l_res = torch.tensor(0.0, device=device)

    loss = final_loss + lambda_step * l_step + lambda_res * l_res + lambda_contrastive * contrastive_loss
    stats = {
        "loss": float(loss.detach().cpu().item()),
        "loss_final": float(final_loss.detach().cpu().item()),
        "loss_step": float(l_step.detach().cpu().item()),
        "loss_res": float(l_res.detach().cpu().item()),
        "loss_contrastive": float(contrastive_loss.detach().cpu().item()),
    }
    return loss, stats, e_prev

In [ ]:
# Full training loop with validation split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
emb_dim = int(batch["sequence"].shape[-1])
model = GRUResidualUpdater(emb_dim=emb_dim, hidden_dim=512, dropout=0.1).to(device)
optim = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

num_epochs = 30
log_every = 100
val_fraction = 0.2
split_generator = torch.Generator().manual_seed(42)

# Checkpoint settings
checkpoint_dir = target_json_path.parent / "checkpoints" / "temporal_residual"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
save_every_epochs = 5
best_metric = float("inf")
best_epoch = -1

if len(records) < 2:
    raise RuntimeError("Need at least 2 records to create a validation split.")

val_size = max(1, int(len(records) * val_fraction))
train_size = len(records) - val_size
if train_size <= 0:
    train_size = len(records) - 1
    val_size = 1

train_dataset, val_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, val_size],
    generator=split_generator,
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    collate_fn=pad_collate,
)
val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0,
    collate_fn=pad_collate,
)

print(f"train records: {len(train_dataset)}")
print(f"val records: {len(val_dataset)}")
print(f"checkpoints: {checkpoint_dir}")

history = []

def evaluate_loss(eval_loader):
    model.eval()
    total_loss = 0.0
    total_final = 0.0
    total_step = 0.0
    total_res = 0.0
    total_contrastive = 0.0
    num_batches = 0

    with torch.no_grad():
        for batch in eval_loader:
            seq = batch["sequence"].to(device)
            target = batch["target"].to(device)
            mask = batch["mask"].to(device)

            loss, stats, _ = residual_temporal_loss(
                model=model,
                seq=seq,
                target=target,
                mask=mask,
                lambda_step=0.4,
                lambda_res=1e-4,
                lambda_contrastive=0.2,
                init_from_first=True,
            )

            total_loss += float(stats["loss"])
            total_final += float(stats["loss_final"])
            total_step += float(stats["loss_step"])
            total_res += float(stats["loss_res"])
            total_contrastive += float(stats["loss_contrastive"])
            num_batches += 1

    if num_batches == 0:
        return None

    return {
        "loss": total_loss / num_batches,
        "loss_final": total_final / num_batches,
        "loss_step": total_step / num_batches,
        "loss_res": total_res / num_batches,
        "loss_contrastive": total_contrastive / num_batches,
    }


def save_checkpoint(path: Path, epoch: int, train_avg: dict, val_avg: Optional[dict]):
    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optim.state_dict(),
        "train": train_avg,
        "val": val_avg,
        "history": history,
        "config": {
            "num_epochs": num_epochs,
            "val_fraction": val_fraction,
            "batch_size": 16,
            "lambda_step": 0.4,
            "lambda_res": 1e-4,
            "lambda_contrastive": 0.2,
            "init_from_first": True,
            "hidden_dim": 512,
        },
    }
    torch.save(payload, path)


for epoch in range(1, num_epochs + 1):
    model.train()

    epoch_loss = 0.0
    epoch_final = 0.0
    epoch_step = 0.0
    epoch_res = 0.0
    epoch_contrastive = 0.0
    num_batches = 0

    progress = tqdm(train_loader, desc=f"epoch {epoch}/{num_epochs}", leave=False)
    for step, batch in enumerate(progress, start=1):
        seq = batch["sequence"].to(device)
        target = batch["target"].to(device)
        mask = batch["mask"].to(device)

        optim.zero_grad(set_to_none=True)
        loss, stats, pred_final = residual_temporal_loss(
            model=model,
            seq=seq,
            target=target,
            mask=mask,
            lambda_step=0.4,
            lambda_res=1e-4,
            lambda_contrastive=0.2,
            init_from_first=True,
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optim.step()

        epoch_loss += float(stats["loss"])
        epoch_final += float(stats["loss_final"])
        epoch_step += float(stats["loss_step"])
        epoch_res += float(stats["loss_res"])
        epoch_contrastive += float(stats["loss_contrastive"])
        num_batches += 1

        progress.set_postfix(
            loss=f"{stats['loss']:.4f}",
            final=f"{stats['loss_final']:.4f}",
            step=f"{stats['loss_step']:.4f}",
            res=f"{stats['loss_res']:.4f}",
            ctr=f"{stats['loss_contrastive']:.4f}",
        )

        if step == 1 or step % log_every == 0:
            print(
                f"epoch {epoch}/{num_epochs} step {step:04d} "
                f"loss={stats['loss']:.4f} final={stats['loss_final']:.4f} "
                f"step={stats['loss_step']:.4f} res={stats['loss_res']:.4f} ctr={stats['loss_contrastive']:.4f}"
            )

    train_avg = {
        "loss": epoch_loss / max(1, num_batches),
        "loss_final": epoch_final / max(1, num_batches),
        "loss_step": epoch_step / max(1, num_batches),
        "loss_res": epoch_res / max(1, num_batches),
        "loss_contrastive": epoch_contrastive / max(1, num_batches),
    }

    val_avg = evaluate_loss(val_loader)

    history.append(
        {
            "epoch": epoch,
            "train": train_avg,
            "val": val_avg,
        }
    )

    print(
        f"[epoch {epoch}/{num_epochs}] "
        f"train_loss={train_avg['loss']:.4f} train_final={train_avg['loss_final']:.4f} "
        f"train_step={train_avg['loss_step']:.4f} train_res={train_avg['loss_res']:.4f} "
        f"train_ctr={train_avg['loss_contrastive']:.4f}"
    )

    if val_avg is not None:
        print(
            f"[epoch {epoch}/{num_epochs}] "
            f"val_loss={val_avg['loss']:.4f} val_final={val_avg['loss_final']:.4f} "
            f"val_step={val_avg['loss_step']:.4f} val_res={val_avg['loss_res']:.4f} "
            f"val_ctr={val_avg['loss_contrastive']:.4f}"
        )
    else:
        print(f"[epoch {epoch}/{num_epochs}] val split empty; skipping validation metrics")

    # Save latest checkpoint every epoch.
    latest_path = checkpoint_dir / "latest.pt"
    save_checkpoint(latest_path, epoch, train_avg, val_avg)

    # Save periodic snapshots.
    if epoch % save_every_epochs == 0:
        epoch_path = checkpoint_dir / f"epoch_{epoch:03d}.pt"
        save_checkpoint(epoch_path, epoch, train_avg, val_avg)

    # Save best checkpoint by validation loss (fallback to train loss if no val split).
    metric = val_avg["loss"] if val_avg is not None else train_avg["loss"]
    if metric < best_metric:
        best_metric = metric
        best_epoch = epoch
        best_path = checkpoint_dir / "best.pt"
        save_checkpoint(best_path, epoch, train_avg, val_avg)
        print(f"saved new best checkpoint: {best_path} (metric={best_metric:.4f})")

final_path = checkpoint_dir / "final.pt"
save_checkpoint(final_path, num_epochs, history[-1]["train"], history[-1]["val"])

print("training complete")
print("last epoch stats:", history[-1])
print(f"best checkpoint epoch: {best_epoch}, metric={best_metric:.4f}")
print(f"latest checkpoint: {checkpoint_dir / 'latest.pt'}")
print(f"final checkpoint: {final_path}")

train records: 30472
val records: 7618
checkpoints: /home/station2/Zermat/RTSGS-SLAM/SceneGraph/checkpoints/temporal_residual


epoch 1/30:   0%|          | 0/1905 [00:00<?, ?it/s]

epoch 1/30 step 0001 loss=1252.9745 final=0.9736 step=1.0010 res=12516005.0000
epoch 1/30 step 0100 loss=0.2472 final=0.1781 step=0.1718 res=3.3056
epoch 1/30 step 0200 loss=0.2267 final=0.1693 step=0.1409 res=10.8847
epoch 1/30 step 0300 loss=0.1893 final=0.1361 step=0.1318 res=4.6499
epoch 1/30 step 0400 loss=0.2190 final=0.1493 step=0.1734 res=3.4375
epoch 1/30 step 0500 loss=0.1875 final=0.1291 step=0.1447 res=5.2596
epoch 1/30 step 0600 loss=0.2141 final=0.1441 step=0.1705 res=18.6684
epoch 1/30 step 0700 loss=0.2148 final=0.1481 step=0.1660 res=2.1922
epoch 1/30 step 0800 loss=0.1611 final=0.1192 step=0.1039 res=3.6388
epoch 1/30 step 0900 loss=0.1934 final=0.1367 step=0.1393 res=10.4291
epoch 1/30 step 1000 loss=0.2088 final=0.1341 step=0.1850 res=7.1279
epoch 1/30 step 1100 loss=0.1953 final=0.1337 step=0.1532 res=3.0746
epoch 1/30 step 1200 loss=0.2001 final=0.1373 step=0.1557 res=5.2217
epoch 1/30 step 1300 loss=0.1827 final=0.1290 step=0.1315 res=10.4194
epoch 1/30 step 1400

epoch 2/30:   0%|          | 0/1905 [00:00<?, ?it/s]

epoch 2/30 step 0001 loss=0.2180 final=0.1594 step=0.1449 res=6.9867
epoch 2/30 step 0100 loss=0.1829 final=0.1286 step=0.1346 res=4.9218
epoch 2/30 step 0200 loss=0.2190 final=0.1501 step=0.1714 res=3.5217
epoch 2/30 step 0300 loss=0.1826 final=0.1311 step=0.1273 res=4.7288
epoch 2/30 step 0400 loss=0.1838 final=0.1391 step=0.1101 res=6.5677
epoch 2/30 step 0500 loss=0.1684 final=0.1320 step=0.0890 res=8.0265
epoch 2/30 step 0600 loss=0.1792 final=0.1237 step=0.1372 res=6.3390
epoch 2/30 step 0700 loss=0.1617 final=0.1173 step=0.1098 res=4.3565
epoch 2/30 step 0800 loss=0.1401 final=0.1094 step=0.0758 res=3.3413
epoch 2/30 step 0900 loss=0.1808 final=0.1315 step=0.1219 res=5.0063
epoch 2/30 step 1000 loss=0.1491 final=0.1125 step=0.0903 res=4.7136
epoch 2/30 step 1100 loss=0.1724 final=0.1403 step=0.0794 res=3.1222
epoch 2/30 step 1200 loss=0.2059 final=0.1450 step=0.1516 res=2.2799
epoch 2/30 step 1300 loss=0.2138 final=0.1453 step=0.1705 res=2.7223
epoch 2/30 step 1400 loss=0.1833 f

...

epoch 7/30:   0%|          | 0/1905 [00:00<?, ?it/s]

epoch 7/30 step 0001 loss=0.1271 final=0.0954 step=0.0788 res=1.9168
epoch 7/30 step 0100 loss=0.1708 final=0.1224 step=0.1209 res=0.5546
epoch 7/30 step 0200 loss=0.1382 final=0.0997 step=0.0952 res=3.7815
epoch 7/30 step 0300 loss=0.1316 final=0.1015 step=0.0747 res=2.8620
epoch 7/30 step 0400 loss=0.1598 final=0.1063 step=0.1322 res=5.6546
epoch 7/30 step 0500 loss=0.1277 final=0.1002 step=0.0673 res=5.1287
epoch 7/30 step 0600 loss=0.1558 final=0.1085 step=0.1177 res=1.2617
epoch 7/30 step 0700 loss=0.2159 final=0.1441 step=0.1795 res=0.1441
epoch 7/30 step 0800 loss=0.1806 final=0.1291 step=0.1280 res=3.2006
epoch 7/30 step 0900 loss=0.2034 final=0.1454 step=0.1447 res=1.1174
epoch 7/30 step 1000 loss=0.1324 final=0.1048 step=0.0682 res=3.7976
epoch 7/30 step 1100 loss=0.1465 final=0.1103 step=0.0900 res=2.2671
epoch 7/30 step 1200 loss=0.1715 final=0.1172 step=0.1357 res=0.4763
epoch 7/30 step 1300 loss=0.1647 final=0.1087 step=0.1400 res=0.1497
epoch 7/30 step 1400 loss=0.1337 f

epoch 8/30:   0%|          | 0/1905 [00:00<?, ?it/s]

epoch 8/30 step 0001 loss=0.1672 final=0.1274 step=0.0992 res=1.5753
epoch 8/30 step 0100 loss=0.1632 final=0.1136 step=0.1227 res=4.5093
epoch 8/30 step 0200 loss=0.1428 final=0.1060 step=0.0918 res=0.5717
epoch 8/30 step 0300 loss=0.1034 final=0.0792 step=0.0585 res=7.4588
epoch 8/30 step 0400 loss=0.1611 final=0.1112 step=0.1246 res=0.7261


KeyboardInterrupt: 